# Project 62 — Local Output Judge Notebook

**Use one local model to critique and score another model's output — fully local LLM-as-judge.**

| Component | Choice |
|-----------|--------|
| LLM runtime | Ollama (qwen3:8b) |
| Framework | LangChain |
| Interface | Jupyter notebook |

## 1 · What You Will Learn

1. How to implement **LLM-as-judge** evaluation patterns locally
2. How to design **scoring rubrics** for different output dimensions
3. How to compare **pairwise outputs** (which response is better?)
4. How to detect **judge bias** and calibrate scores
5. How to aggregate judgments into **actionable quality metrics**

## 2 · Why This Project Matters

Human evaluation is expensive and slow. LLM-as-judge provides fast, consistent
scoring that correlates well with human preferences. Running the judge locally
means you can evaluate thousands of outputs without API costs.

## 3 · Environment Setup

In [ ]:
# !pip install -q langchain langchain-ollama pandas

## 4 · Imports and Configuration

In [ ]:
import json
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = 'qwen3:8b'
generator = ChatOllama(model=MODEL, temperature=0.7)
judge = ChatOllama(model=MODEL, temperature=0.1)
print(f'Generator & Judge ready: {MODEL}')

## 5 · Generate Test Outputs

We generate two responses per question using different system prompts.

In [ ]:
QUESTIONS = [
    'What are the pros and cons of remote work?',
    'Explain machine learning to a 10-year-old.',
    'Write a professional email declining a meeting.',
    'Summarize key principles of agile development.',
]

prompt_a = ChatPromptTemplate.from_messages([('human', '{q}')])
prompt_b = ChatPromptTemplate.from_messages([
    ('system', 'You are a concise expert. Give structured answers with bullet points.'),
    ('human', '{q}'),
])
chain_a = prompt_a | generator | StrOutputParser()
chain_b = prompt_b | generator | StrOutputParser()

test_pairs = []
for q in QUESTIONS:
    print(f'Generating for: {q[:50]}...')
    resp_a = chain_a.invoke({'q': q})
    resp_b = chain_b.invoke({'q': q})
    test_pairs.append({'question': q, 'response_a': resp_a, 'response_b': resp_b})
print(f'Generated {len(test_pairs)} response pairs')

## 6 · Build Pointwise and Pairwise Judges

In [ ]:
def pointwise_judge(question, response):
    prompt = ChatPromptTemplate.from_messages([
        ('system', 'Score 1-5. Return ONLY JSON: {"helpfulness": N, "accuracy": N, "clarity": N, "overall": N, "reasoning": "..."}'),
        ('human', 'Question: {question}\nResponse: {response}'),
    ])
    raw = (prompt | judge | StrOutputParser()).invoke({'question': question, 'response': response})
    try:
        s, e = raw.find('{'), raw.rfind('}') + 1
        if s >= 0 and e > s: return json.loads(raw[s:e])
    except: pass
    return {'overall': 3, 'reasoning': 'Parse error'}

def pairwise_judge(question, resp_a, resp_b):
    prompt = ChatPromptTemplate.from_messages([
        ('system', 'Compare A and B. Return ONLY JSON: {"winner": "A"/"B"/"tie", "confidence": 1-5, "reasoning": "..."}'),
        ('human', 'Question: {question}\nResponse A: {resp_a}\nResponse B: {resp_b}'),
    ])
    raw = (prompt | judge | StrOutputParser()).invoke({'question': question, 'resp_a': resp_a, 'resp_b': resp_b})
    try:
        s, e = raw.find('{'), raw.rfind('}') + 1
        if s >= 0 and e > s: return json.loads(raw[s:e])
    except: pass
    return {'winner': 'tie', 'confidence': 1, 'reasoning': 'Parse error'}

print('Judge functions ready.')

## 7 · Run Evaluations

In [ ]:
eval_results = []
for pair in test_pairs:
    q = pair['question']
    print(f'Judging: {q[:50]}...')
    sa = pointwise_judge(q, pair['response_a'])
    sb = pointwise_judge(q, pair['response_b'])
    cmp = pairwise_judge(q, pair['response_a'], pair['response_b'])
    eval_results.append({'question': q, 'score_a': sa, 'score_b': sb, 'pairwise': cmp})
    print(f'  A={sa.get("overall","?")}, B={sb.get("overall","?")}, Winner={cmp.get("winner","?")}')
print(f'Completed {len(eval_results)} evaluations')

## 8 · Analyze Results

In [ ]:
rows = [{'question': er['question'][:40],
         'A_overall': er['score_a'].get('overall', 0),
         'B_overall': er['score_b'].get('overall', 0),
         'winner': er['pairwise'].get('winner', '?')}
        for er in eval_results]
df = pd.DataFrame(rows)
print('EVALUATION SUMMARY')
print(df.to_string(index=False))
print(f'\nA wins: {sum(1 for r in rows if r["winner"]=="A")}, '
      f'B wins: {sum(1 for r in rows if r["winner"]=="B")}')

## 9 · Bias Detection (Position Swap)

In [ ]:
pair = test_pairs[0]
fwd = pairwise_judge(pair['question'], pair['response_a'], pair['response_b'])
rev = pairwise_judge(pair['question'], pair['response_b'], pair['response_a'])
print(f'Forward: winner={fwd.get("winner","?")}')
print(f'Reverse: winner={rev.get("winner","?")}')
# Check consistency after adjusting for swap
rev_adj = {'A': 'B', 'B': 'A'}.get(rev.get('winner',''), 'tie')
print(f'Consistent: {fwd.get("winner") == rev_adj or "tie" in [fwd.get("winner"), rev_adj]}')

## 10 · Save Results

In [ ]:
with open('judge_results.json', 'w') as f:
    json.dump(eval_results, f, indent=2, default=str)
print('Results saved.')

## 11 · Limitations

- **Self-judging bias** — same model family may favor its own style
- **Position bias** — judge may prefer whichever response it reads first
- **No ground truth** — no human scores to calibrate against
- Scoring can be noisy with small local models

## 12 · How to Improve

1. Use a different model as judge
2. Add human calibration
3. Ensemble judging with multiple prompts
4. Position debiasing (run both orderings)

## 13 · Mini Challenge

1. Add a third prompt variant and do round-robin comparison
2. Implement position debiasing by averaging forward/reverse
3. Add dimension-specific analysis

## 14 · Key Takeaways

- Pointwise scoring gives absolute quality; pairwise gives relative preference
- Position bias is real and testable via swap experiments
- LLM-as-judge enables fast evaluation without human annotators
- Local evaluation with Ollama keeps costs zero